# KPI Analysis and A/B Testing

## Objective

The purpose of this notebook is to evaluate whether Vanguard's new digital interface performed better than the traditional interface.

The analysis compares the `Test` and `Control` groups using a focused set of KPIs related to completion, friction, usability, and efficiency.

The main dataset used in this notebook is `session_ab_df`, which contains one row per session and only includes users from the Test and Control groups. A secondary dataset, `client_ab_df`, is also loaded for client-level checks and demographic context.

---

## Analysis Strategy

The main analysis focuses on sessions that actually started the digital process.

This avoids judging the interface using sessions where the user never entered the process. For time and efficiency metrics, only completed sessions are used, because completion time and number of events required to complete are only meaningful when the process was actually completed.

The analysis focuses on seven KPIs:

| KPI | Purpose |
|---|---|
| Completion Rate | Measures whether users completed the process |
| Completion Rate with 5% Threshold | Checks whether the improvement is large enough to be meaningful |
| Friction Rate | Measures non-linear or problematic journeys |
| Backtracking Rate | Measures whether users moved backwards |
| Repeated Steps Rate | Measures whether users repeated steps |
| Median Completion Time | Measures how long completed sessions took |
| Median Events per Completed Session | Measures how many interactions were needed to complete |

Outlier treatment is only considered for continuous metrics, such as time and number of events. Proportion-based KPIs are not filtered for outliers.

---

## Notebook Structure

1. Introduction
2. Imports and data loading
3. Analysis base
4. Helper functions
5. KPI tests
6. Final KPI results table
7. Save final A/B summary CSV
8. Funnel and drop-off analysis
9. Final business conclusion

In [48]:
import pandas as pd
import numpy as np

from pathlib import Path
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest

In [2]:
input_path = Path("../1.2_Merged_Files_For_EDA/")
output_path = Path("../1.3_KPI_AB_Testing_Outputs/")

output_path.mkdir(parents=True, exist_ok=True)

In [4]:
session_ab_df = pd.read_csv(input_path / "session_ab_df.csv")
client_ab_df = pd.read_csv(input_path / "client_ab_df.csv")

In [6]:
date_cols = ["session_start", "session_end", "first_start_time", "first_confirm_time"]

for col in date_cols:
    if col in session_ab_df.columns:
        session_ab_df[col] = pd.to_datetime(session_ab_df[col], errors="coerce")

bool_cols = [
    "started", "completed", "has_backtracking", "has_step_jump", "has_repeated_step_event",
    "has_repeated_steps", "straight_through", "friction_flag", "incomplete_flag",
    "did_not_start_with_start", "did_not_end_with_confirm"
]

for col in bool_cols:
    if col in session_ab_df.columns:
        session_ab_df[col] = session_ab_df[col].astype(bool)

In [7]:
print("session_ab_df shape:", session_ab_df.shape)
print("client_ab_df shape:", client_ab_df.shape)

print("\nSession-level variation counts:")
print(session_ab_df["variation"].value_counts())

print("\nClient-level variation counts:")
print(client_ab_df["variation"].value_counts())

print("\nUnique clients in session_ab_df:")
print(session_ab_df.groupby("variation")["client_id"].nunique())

session_ab_df shape: (69447, 34)
client_ab_df shape: (50500, 39)

Session-level variation counts:
variation
Test       37204
Control    32243
Name: count, dtype: int64

Client-level variation counts:
variation
Test       26968
Control    23532
Name: count, dtype: int64

Unique clients in session_ab_df:
variation
Control    23532
Test       26968
Name: client_id, dtype: int64


In [8]:
started_sessions_df = session_ab_df[session_ab_df["started"]].copy()
completed_sessions_df = started_sessions_df[started_sessions_df["completed"]].copy()

print("All A/B sessions:", len(session_ab_df))
print("Started A/B sessions:", len(started_sessions_df))
print("Completed started sessions:", len(completed_sessions_df))

print("\nStarted sessions by variation:")
print(started_sessions_df["variation"].value_counts())

print("\nCompleted sessions by variation:")
print(completed_sessions_df["variation"].value_counts())

All A/B sessions: 69447
Started A/B sessions: 64181
Completed started sessions: 32903

Started sessions by variation:
variation
Test       33219
Control    30962
Name: count, dtype: int64

Completed sessions by variation:
variation
Test       17961
Control    14942
Name: count, dtype: int64


## Analysis Base

The main analysis focuses on sessions that actually started the digital process.

This avoids evaluating the interface using sessions where the user never entered the process. For this reason, proportion-based KPIs such as completion rate, friction rate, backtracking rate, and repeated steps rate are calculated using started sessions.

For time and efficiency KPIs, only completed sessions are used, because completion time and number of events required to complete are only meaningful when the process reached the confirmation step.

In [9]:
started_check = (
    session_ab_df
    .groupby("variation")
    .agg(
        total_sessions=("visit_id", "count"),
        started_sessions=("started", "sum"),
        started_rate=("started", "mean")
    )
    .round(4)
)

started_check

,total_sessions,started_sessions,started_rate
variation,,,
Control,32243,30962,0.9603
Test,37204,33219,0.8929


The started session rate is checked before the KPI analysis to understand how much of the available A/B session data actually entered the digital process.

The main KPI analysis will focus on started sessions, since these sessions are the most relevant for evaluating the user experience of the interface itself.

Now we are going to define the helper functions for the KPIs:


In [10]:
def proportion_kpi_test(df, metric_col, success_label, alternative="larger"):
    summary = (
        df
        .groupby("variation")
        .agg(
            sessions=("visit_id", "count"),
            successes=(metric_col, "sum"),
            rate=(metric_col, "mean")
        )
    )

    control_successes = summary.loc["Control", "successes"]
    test_successes = summary.loc["Test", "successes"]

    control_sessions = summary.loc["Control", "sessions"]
    test_sessions = summary.loc["Test", "sessions"]

    count = np.array([test_successes, control_successes])
    nobs = np.array([test_sessions, control_sessions])

    z_stat, p_value = proportions_ztest(count=count, nobs=nobs, alternative=alternative)

    test_rate = summary.loc["Test", "rate"]
    control_rate = summary.loc["Control", "rate"]
    difference = test_rate - control_rate

    result = pd.DataFrame({
        "kpi": [success_label],
        "test_rate": [test_rate],
        "control_rate": [control_rate],
        "difference": [difference],
        "z_stat": [z_stat],
        "p_value": [p_value],
        "significant_0.05": [p_value < 0.05]
    })

    summary = summary.reset_index()
    summary["rate"] = summary["rate"].round(4)

    result[["test_rate", "control_rate", "difference", "z_stat", "p_value"]] = (
        result[["test_rate", "control_rate", "difference", "z_stat", "p_value"]].round(4)
    )

    return summary, result

In [11]:
def continuous_kpi_test(df, metric_col, kpi_name, alternative="less"):
    control_values = df.loc[df["variation"] == "Control", metric_col].dropna()
    test_values = df.loc[df["variation"] == "Test", metric_col].dropna()

    u_stat, p_value = stats.mannwhitneyu(test_values, control_values, alternative=alternative)

    summary = (
        df
        .groupby("variation")
        .agg(
            sessions=(metric_col, "count"),
            mean_value=(metric_col, "mean"),
            median_value=(metric_col, "median")
        )
        .round(4)
        .reset_index()
    )

    result = pd.DataFrame({
        "kpi": [kpi_name],
        "test_median": [test_values.median()],
        "control_median": [control_values.median()],
        "median_difference": [test_values.median() - control_values.median()],
        "u_stat": [u_stat],
        "p_value": [p_value],
        "significant_0.05": [p_value < 0.05]
    })

    result[["test_median", "control_median", "median_difference", "u_stat", "p_value"]] = (
        result[["test_median", "control_median", "median_difference", "u_stat", "p_value"]].round(4)
    )

    return summary, result

## Testing Functions

Two helper functions were created to keep the analysis consistent and avoid repeating code.

For proportion-based KPIs, such as completion rate and backtracking rate, a two-proportion z-test is used.

For continuous KPIs, such as completion time and number of events, a Mann-Whitney U test is used because these variables are likely to be skewed and affected by extreme values.

## KPI 1 — Completion Rate

Completion Rate measures the percentage of started sessions that reached the confirmation step.

This is the primary success metric of the experiment because the main goal of the new interface was to help more users complete the digital process.

**Hypothesis**

- H0: Completion Rate Test <= Completion Rate Control
- H1: Completion Rate Test > Completion Rate Control

In [12]:
completion_summary, completion_result = proportion_kpi_test(
    df=started_sessions_df,
    metric_col="completed",
    success_label="Completion Rate",
    alternative="larger"
)

display(completion_summary)
display(completion_result)

,variation,sessions,successes,rate
0,Control,30962,14942,0.4826
1,Test,33219,17961,0.5407


,kpi,test_rate,control_rate,difference,z_stat,p_value,significant_0.05
0,Completion Rate,0.5407,0.4826,0.0581,14.7129,0.0,True


In [13]:
hypothesis_results = []

completion_result["test_type"] = "Two-proportion z-test"
completion_result["alternative"] = "Test > Control"
completion_result["decision"] = np.where(
    completion_result["significant_0.05"],
    "Reject H0",
    "Fail to reject H0"
)

hypothesis_results.append(completion_result)

In [14]:
completion_test_rate = completion_result.loc[0, "test_rate"]
completion_control_rate = completion_result.loc[0, "control_rate"]
completion_diff = completion_result.loc[0, "difference"]
completion_p = completion_result.loc[0, "p_value"]
completion_significant = completion_result.loc[0, "significant_0.05"]

if completion_significant and completion_diff > 0:
    print(f"The Test group achieved a higher completion rate than the Control group.")
    print(f"Test: {completion_test_rate:.2%} | Control: {completion_control_rate:.2%} | Difference: {completion_diff:.2%}")
    print(f"The difference is statistically significant at alpha = 0.05, with p-value = {completion_p:.4f}.")
else:
    print(f"The Test group did not show a statistically significant improvement in completion rate.")
    print(f"Test: {completion_test_rate:.2%} | Control: {completion_control_rate:.2%} | Difference: {completion_diff:.2%}")
    print(f"p-value = {completion_p:.4f}.")

The Test group achieved a higher completion rate than the Control group.
Test: 54.07% | Control: 48.26% | Difference: 5.81%
The difference is statistically significant at alpha = 0.05, with p-value = 0.0000.


### Interpretation

The Test group achieved a higher Completion Rate than the Control group.

The Test group completed 54.07% of started sessions, compared with 48.26% in the Control group. This represents an improvement of 5.81 percentage points.

The difference is statistically significant at alpha = 0.05, with p-value < 0.0001.

This supports the idea that the redesigned interface was more effective at helping users complete the digital process.

## KPI 2 — Completion Rate with 5% Threshold

This KPI checks whether the Test group improved Completion Rate by more than 5 percentage points compared with the Control group.

This is useful because a statistically significant improvement may still be too small to justify the cost or effort of a redesign.

**Hypothesis**

- H0: Completion Rate Test - Completion Rate Control <= 5%
- H1: Completion Rate Test - Completion Rate Control > 5%

In [15]:
# =========================
# Completion Rate with 5% threshold
# =========================

threshold = 0.05

completion_threshold_summary = (
    started_sessions_df
    .groupby("variation")
    .agg(
        sessions=("visit_id", "count"),
        completed_sessions=("completed", "sum"),
        completion_rate=("completed", "mean")
    )
    .round(4)
)

test_completed = completion_threshold_summary.loc["Test", "completed_sessions"]
control_completed = completion_threshold_summary.loc["Control", "completed_sessions"]

test_sessions = completion_threshold_summary.loc["Test", "sessions"]
control_sessions = completion_threshold_summary.loc["Control", "sessions"]

count = np.array([test_completed, control_completed])
nobs = np.array([test_sessions, control_sessions])

z_stat, p_value = proportions_ztest(
    count=count,
    nobs=nobs,
    value=threshold,
    alternative="larger"
)

test_rate = completion_threshold_summary.loc["Test", "completion_rate"]
control_rate = completion_threshold_summary.loc["Control", "completion_rate"]
difference = test_rate - control_rate

completion_threshold_result = pd.DataFrame({
    "kpi": ["Completion Rate with 5% Threshold"],
    "test_rate": [test_rate],
    "control_rate": [control_rate],
    "difference": [difference],
    "threshold": [threshold],
    "z_stat": [z_stat],
    "p_value": [p_value],
    "significant_0.05": [p_value < 0.05]
})

completion_threshold_result[["test_rate", "control_rate", "difference", "threshold", "z_stat", "p_value"]] = (
    completion_threshold_result[["test_rate", "control_rate", "difference", "threshold", "z_stat", "p_value"]].round(4)
)

display(completion_threshold_summary)
display(completion_threshold_result)

,sessions,completed_sessions,completion_rate
variation,,,
Control,30962,14942,0.4826
Test,33219,17961,0.5407


,kpi,test_rate,control_rate,difference,threshold,z_stat,p_value,significant_0.05
0,Completion Rate with 5% Threshold,0.5407,0.4826,0.0581,0.05,2.0497,0.0202,True


In [16]:
completion_threshold_result["test_type"] = "Two-proportion z-test with 5% threshold"
completion_threshold_result["alternative"] = "Test - Control > 5%"
completion_threshold_result["decision"] = np.where(
    completion_threshold_result["significant_0.05"],
    "Reject H0",
    "Fail to reject H0"
)

hypothesis_results.append(completion_threshold_result)

In [17]:
threshold_test_rate = completion_threshold_result.loc[0, "test_rate"]
threshold_control_rate = completion_threshold_result.loc[0, "control_rate"]
threshold_diff = completion_threshold_result.loc[0, "difference"]
threshold_p = completion_threshold_result.loc[0, "p_value"]
threshold_significant = completion_threshold_result.loc[0, "significant_0.05"]

if threshold_significant and threshold_diff > threshold:
    print("The Test group improved Completion Rate by more than the 5% threshold.")
    print(f"Test: {threshold_test_rate:.2%} | Control: {threshold_control_rate:.2%} | Difference: {threshold_diff:.2%}")
    print(f"The threshold test is statistically significant at alpha = 0.05, with p-value = {threshold_p:.4f}.")
else:
    print("The Test group did not show enough evidence of improving Completion Rate by more than 5%.")
    print(f"Test: {threshold_test_rate:.2%} | Control: {threshold_control_rate:.2%} | Difference: {threshold_diff:.2%}")
    print(f"p-value = {threshold_p:.4f}.")

The Test group improved Completion Rate by more than the 5% threshold.
Test: 54.07% | Control: 48.26% | Difference: 5.81%
The threshold test is statistically significant at alpha = 0.05, with p-value = 0.0202.


### Interpretation

The Test group improved Completion Rate by 5.81 percentage points compared with the Control group.

This improvement is above the 5% business threshold, and the threshold test is statistically significant at alpha = 0.05, with p-value = 0.0202.

This suggests that the redesign did not only improve completion statistically, but also reached the minimum practical improvement defined for the experiment.

However, the improvement is only slightly above the 5% threshold, so the result should be considered positive but not overwhelmingly strong.

## KPI 3 — Friction Rate

Friction Rate measures the percentage of started sessions that did not follow the ideal process flow.

A session is considered to have friction if it includes abandonment, repeated steps, backtracking, skipped steps, or another non-linear journey.

This KPI helps evaluate whether the new interface created a smoother user experience.

**Hypothesis**

- H0: Friction Rate Test >= Friction Rate Control
- H1: Friction Rate Test < Friction Rate Control

In [18]:
friction_summary, friction_result = proportion_kpi_test(
    df=started_sessions_df,
    metric_col="friction_flag",
    success_label="Friction Rate",
    alternative="smaller"
)

display(friction_summary)
display(friction_result)

,variation,sessions,successes,rate
0,Control,30962,21480,0.6938
1,Test,33219,22606,0.6805


,kpi,test_rate,control_rate,difference,z_stat,p_value,significant_0.05
0,Friction Rate,0.6805,0.6938,-0.0132,-3.614,0.0002,True


In [19]:
friction_result["test_type"] = "Two-proportion z-test"
friction_result["alternative"] = "Test < Control"
friction_result["decision"] = np.where(
    friction_result["significant_0.05"],
    "Reject H0",
    "Fail to reject H0"
)

hypothesis_results.append(friction_result)

In [20]:
friction_test_rate = friction_result.loc[0, "test_rate"]
friction_control_rate = friction_result.loc[0, "control_rate"]
friction_diff = friction_result.loc[0, "difference"]
friction_p = friction_result.loc[0, "p_value"]
friction_significant = friction_result.loc[0, "significant_0.05"]

if friction_significant and friction_diff < 0:
    print("The Test group showed a significantly lower friction rate than the Control group.")
    print(f"Test: {friction_test_rate:.2%} | Control: {friction_control_rate:.2%} | Difference: {friction_diff:.2%}")
    print(f"The difference is statistically significant at alpha = 0.05, with p-value = {friction_p:.4f}.")
else:
    print("The Test group did not show a statistically significant reduction in friction rate.")
    print(f"Test: {friction_test_rate:.2%} | Control: {friction_control_rate:.2%} | Difference: {friction_diff:.2%}")
    print(f"p-value = {friction_p:.4f}.")

The Test group showed a significantly lower friction rate than the Control group.
Test: 68.05% | Control: 69.38% | Difference: -1.32%
The difference is statistically significant at alpha = 0.05, with p-value = 0.0002.


### Interpretation

The Test group showed a slightly lower Friction Rate than the Control group.

The Test group had friction in 68.05% of started sessions, compared with 69.38% in the Control group. This represents a reduction of 1.32 percentage points.

The difference is statistically significant at alpha = 0.05, with p-value = 0.0002.

This suggests that the redesigned interface slightly reduced non-linear or problematic journeys. However, the effect size is small, and friction remains high in both groups. Therefore, while the Test version performs better on this KPI, the overall user journey still shows considerable room for improvement.

## KPI 4 — Backtracking Rate

Backtracking Rate measures the percentage of started sessions where users moved backwards in the process.

This KPI is useful because moving backwards can indicate confusion, hesitation, or difficulty understanding the interface.

**Hypothesis**

- H0: Backtracking Rate Test >= Backtracking Rate Control
- H1: Backtracking Rate Test < Backtracking Rate Control

In [21]:
backtracking_summary, backtracking_result = proportion_kpi_test(
    df=started_sessions_df,
    metric_col="has_backtracking",
    success_label="Backtracking Rate",
    alternative="smaller"
)

display(backtracking_summary)
display(backtracking_result)

,variation,sessions,successes,rate
0,Control,30962,6368,0.2057
1,Test,33219,9889,0.2977


,kpi,test_rate,control_rate,difference,z_stat,p_value,significant_0.05
0,Backtracking Rate,0.2977,0.2057,0.092,26.7852,1.0,False


In [22]:
backtracking_result["test_type"] = "Two-proportion z-test"
backtracking_result["alternative"] = "Test < Control"
backtracking_result["decision"] = np.where(
    backtracking_result["significant_0.05"],
    "Reject H0",
    "Fail to reject H0"
)

hypothesis_results.append(backtracking_result)

In [24]:
backtracking_test_rate = backtracking_result.loc[0, "test_rate"]
backtracking_control_rate = backtracking_result.loc[0, "control_rate"]
backtracking_diff = backtracking_result.loc[0, "difference"]
backtracking_p = backtracking_result.loc[0, "p_value"]
backtracking_significant = backtracking_result.loc[0, "significant_0.05"]

if backtracking_significant and backtracking_diff < 0:
    print("The Test group showed a significantly lower backtracking rate than the Control group.")
    print(f"Test: {backtracking_test_rate:.2%} | Control: {backtracking_control_rate:.2%} | Difference: {backtracking_diff:.2%}")
    print(f"The difference is statistically significant at alpha = 0.05, with p-value = {backtracking_p:.4f}.")

elif backtracking_diff > 0:
    print("The Test group did not reduce backtracking rate.")
    print(f"Test: {backtracking_test_rate:.2%} | Control: {backtracking_control_rate:.2%} | Difference: {backtracking_diff:.2%}")
    print(f"The result goes in the opposite direction of the hypothesis, with p-value = {backtracking_p:.4f}.")
    print("This suggests that the new interface may have caused more users to move backwards during the process.")

else:
    print("The Test group did not show a statistically significant reduction in backtracking rate.")
    print(f"Test: {backtracking_test_rate:.2%} | Control: {backtracking_control_rate:.2%} | Difference: {backtracking_diff:.2%}")
    print(f"p-value = {backtracking_p:.4f}.")

The Test group did not reduce backtracking rate.
Test: 29.77% | Control: 20.57% | Difference: 9.20%
The result goes in the opposite direction of the hypothesis, with p-value = 1.0000.
This suggests that the new interface may have caused more users to move backwards during the process.


### Interpretation

The Test group did not reduce Backtracking Rate compared with the Control group.

In fact, the Test group showed a higher Backtracking Rate: 29.77% compared with 20.57% in the Control group. This represents an increase of 9.20 percentage points.

Because the hypothesis tested whether Test had a lower Backtracking Rate, the result does not support rejecting the null hypothesis.

This suggests that although the new interface may help more users complete the process, it may also lead to more users moving backwards during the journey, which could indicate confusion or a less smooth user experience.


## KPI 5 — Repeated Steps Rate

Repeated Steps Rate measures the percentage of started sessions where users repeated one or more process steps.

This KPI helps identify whether users needed to revisit the same step multiple times, which may indicate hesitation, confusion, or inefficiency in the process.

**Hypothesis**

- H0: Repeated Steps Rate Test >= Repeated Steps Rate Control
- H1: Repeated Steps Rate Test < Repeated Steps Rate Control

In [25]:
repeated_steps_summary, repeated_steps_result = proportion_kpi_test(
    df=started_sessions_df,
    metric_col="has_repeated_steps",
    success_label="Repeated Steps Rate",
    alternative="smaller"
)

display(repeated_steps_summary)
display(repeated_steps_result)

,variation,sessions,successes,rate
0,Control,30962,11703,0.3780
1,Test,33219,15728,0.4735


,kpi,test_rate,control_rate,difference,z_stat,p_value,significant_0.05
0,Repeated Steps Rate,0.4735,0.378,0.0955,24.434,1.0,False


In [26]:
repeated_steps_result["test_type"] = "Two-proportion z-test"
repeated_steps_result["alternative"] = "Test < Control"
repeated_steps_result["decision"] = np.where(
    repeated_steps_result["significant_0.05"],
    "Reject H0",
    "Fail to reject H0"
)

hypothesis_results.append(repeated_steps_result)

In [27]:
repeated_test_rate = repeated_steps_result.loc[0, "test_rate"]
repeated_control_rate = repeated_steps_result.loc[0, "control_rate"]
repeated_diff = repeated_steps_result.loc[0, "difference"]
repeated_p = repeated_steps_result.loc[0, "p_value"]
repeated_significant = repeated_steps_result.loc[0, "significant_0.05"]

if repeated_significant and repeated_diff < 0:
    print("The Test group showed a significantly lower repeated steps rate than the Control group.")
    print(f"Test: {repeated_test_rate:.2%} | Control: {repeated_control_rate:.2%} | Difference: {repeated_diff:.2%}")
    print(f"The difference is statistically significant at alpha = 0.05, with p-value = {repeated_p:.4f}.")

elif repeated_diff > 0:
    print("The Test group did not reduce repeated steps rate.")
    print(f"Test: {repeated_test_rate:.2%} | Control: {repeated_control_rate:.2%} | Difference: {repeated_diff:.2%}")
    print(f"The result goes in the opposite direction of the hypothesis, with p-value = {repeated_p:.4f}.")
    print("This suggests that users in the Test group repeated steps more often during the process.")

else:
    print("The Test group did not show a statistically significant reduction in repeated steps rate.")
    print(f"Test: {repeated_test_rate:.2%} | Control: {repeated_control_rate:.2%} | Difference: {repeated_diff:.2%}")
    print(f"p-value = {repeated_p:.4f}.")

The Test group did not reduce repeated steps rate.
Test: 47.35% | Control: 37.80% | Difference: 9.55%
The result goes in the opposite direction of the hypothesis, with p-value = 1.0000.
This suggests that users in the Test group repeated steps more often during the process.


### Interpretation

The Test group did not reduce Repeated Steps Rate compared with the Control group.

In fact, the Test group showed a higher Repeated Steps Rate: 47.35% compared with 37.80% in the Control group. This represents an increase of 9.55 percentage points.

Because the hypothesis tested whether Test had a lower Repeated Steps Rate, the result does not support rejecting the null hypothesis.

This suggests that users in the Test group repeated steps more often during the process, which may indicate that the redesigned interface created more repeated interactions or uncertainty, even though it improved overall completion.

## KPI 6 — Median Completion Time

Median Completion Time measures how long completed sessions took from the first `start` step to the first `confirm` step.

This KPI helps evaluate whether the new interface allowed users to complete the process faster.

Only completed sessions are used for this metric.

**Hypothesis**

- H0: Completion Time Test >= Completion Time Control
- H1: Completion Time Test < Completion Time Control

In [31]:
# =========================
# Completion time outlier check
# =========================

completion_time_df = completed_sessions_df.dropna(subset=["completion_time_min"]).copy()

q1 = completion_time_df["completion_time_min"].quantile(0.25)
q3 = completion_time_df["completion_time_min"].quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

completion_time_iqr_df = completion_time_df[
    (completion_time_df["completion_time_min"] >= lower_bound) &
    (completion_time_df["completion_time_min"] <= upper_bound)
].copy()

removed_rows = len(completion_time_df) - len(completion_time_iqr_df)
removed_pct = removed_rows / len(completion_time_df)

print("Original rows:", len(completion_time_df))
print("Rows after IQR filter:", len(completion_time_iqr_df))
print("Removed rows:", removed_rows)
print(f"Removed percentage: {removed_pct:.2%}")
print(f"IQR upper bound: {upper_bound:.2f} minutes")

Original rows: 32903
Rows after IQR filter: 30177
Removed rows: 2726
Removed percentage: 8.28%
IQR upper bound: 14.22 minutes


In [29]:
completion_time_summary, completion_time_result = continuous_kpi_test(
    df=completion_time_df,
    metric_col="completion_time_min",
    kpi_name="Median Completion Time",
    alternative="less"
)

display(completion_time_summary)
display(completion_time_result)

,variation,sessions,mean_value,median_value
0,Control,14942,6.5033,4.5167
1,Test,17961,6.2193,3.9500


,kpi,test_median,control_median,median_difference,u_stat,p_value,significant_0.05
0,Median Completion Time,3.95,4.5167,-0.5667,121519681.0,0.0,True


In [32]:
completion_time_result["test_type"] = "Mann-Whitney U test"
completion_time_result["alternative"] = "Test < Control"
completion_time_result["decision"] = np.where(
    completion_time_result["significant_0.05"],
    "Reject H0",
    "Fail to reject H0"
)

completion_time_result["observed_direction"] = np.where(
    completion_time_result["median_difference"] < 0,
    "Test < Control",
    np.where(completion_time_result["median_difference"] > 0, "Test > Control", "Test = Control")
)

hypothesis_results.append(completion_time_result)

In [33]:
time_test_median = completion_time_result.loc[0, "test_median"]
time_control_median = completion_time_result.loc[0, "control_median"]
time_diff = completion_time_result.loc[0, "median_difference"]
time_p = completion_time_result.loc[0, "p_value"]
time_significant = completion_time_result.loc[0, "significant_0.05"]

if time_significant and time_diff < 0:
    print("The Test group completed the process significantly faster than the Control group.")
    print(f"Test median: {time_test_median:.2f} min | Control median: {time_control_median:.2f} min | Difference: {time_diff:.2f} min")
    print(f"This represents a reduction of approximately {abs(time_diff * 60):.0f} seconds.")
    print("The difference is statistically significant at alpha = 0.05, with p-value < 0.0001.")

elif time_diff > 0:
    print("The Test group did not reduce completion time.")
    print(f"Test median: {time_test_median:.2f} min | Control median: {time_control_median:.2f} min | Difference: {time_diff:.2f} min")
    print(f"The result goes in the opposite direction of the hypothesis, with p-value = {time_p:.4f}.")

else:
    print("The Test group did not show a statistically significant reduction in completion time.")
    print(f"Test median: {time_test_median:.2f} min | Control median: {time_control_median:.2f} min | Difference: {time_diff:.2f} min")
    print(f"p-value = {time_p:.4f}.")

The Test group completed the process significantly faster than the Control group.
Test median: 3.95 min | Control median: 4.52 min | Difference: -0.57 min
This represents a reduction of approximately 34 seconds.
The difference is statistically significant at alpha = 0.05, with p-value < 0.0001.


### Interpretation

The Test group completed the process faster than the Control group.

The median completion time was 3.95 minutes for Test and 4.52 minutes for Control, meaning that completed Test sessions were around 0.57 minutes faster, approximately 34 seconds.

The Mann-Whitney U test shows that this difference is statistically significant at alpha = 0.05, with p-value < 0.0001.

An IQR outlier check was also performed. The filter would remove 8.28% of completed sessions, which is above the 5% threshold defined for this project. For that reason, the IQR-filtered dataset was not used as the main result.

Instead, the raw data was kept and the median was used as the main metric because it is more robust to extreme values than the mean.

## KPI 7 — Median Events per Completed Session

Median Events per Completed Session measures how many web interactions were needed to complete the process.

This KPI helps evaluate process efficiency. If users in the Test group need fewer events to complete the process, the new interface may be more efficient.

Only completed sessions are used for this metric.

**Hypothesis**

- H0: Events per Completed Session Test >= Events per Completed Session Control
- H1: Events per Completed Session Test < Events per Completed Session Control

In [34]:
# =========================
# Events per completed session outlier check
# =========================

events_completed_df = completed_sessions_df.dropna(subset=["n_events"]).copy()

q1 = events_completed_df["n_events"].quantile(0.25)
q3 = events_completed_df["n_events"].quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

events_completed_iqr_df = events_completed_df[
    (events_completed_df["n_events"] >= lower_bound) &
    (events_completed_df["n_events"] <= upper_bound)
].copy()

removed_rows = len(events_completed_df) - len(events_completed_iqr_df)
removed_pct = removed_rows / len(events_completed_df)

print("Original rows:", len(events_completed_df))
print("Rows after IQR filter:", len(events_completed_iqr_df))
print("Removed rows:", removed_rows)
print(f"Removed percentage: {removed_pct:.2%}")
print(f"IQR upper bound: {upper_bound:.2f} events")

Original rows: 32903
Rows after IQR filter: 31458
Removed rows: 1445
Removed percentage: 4.39%
IQR upper bound: 10.00 events


In [35]:
events_completed_summary, events_completed_result = continuous_kpi_test(
    df=events_completed_df,
    metric_col="n_events",
    kpi_name="Median Events per Completed Session",
    alternative="less"
)

display(events_completed_summary)
display(events_completed_result)

,variation,sessions,mean_value,median_value
0,Control,14942,6.0371,5.0
1,Test,17961,6.1845,5.0


,kpi,test_median,control_median,median_difference,u_stat,p_value,significant_0.05
0,Median Events per Completed Session,5.0,5.0,0.0,140078644.5,1.0,False


In [36]:
events_completed_result["test_type"] = "Mann-Whitney U test"
events_completed_result["alternative"] = "Test < Control"
events_completed_result["decision"] = np.where(
    events_completed_result["significant_0.05"],
    "Reject H0",
    "Fail to reject H0"
)

events_completed_result["observed_direction"] = np.where(
    events_completed_result["median_difference"] < 0,
    "Test < Control",
    np.where(events_completed_result["median_difference"] > 0, "Test > Control", "Test = Control")
)

hypothesis_results.append(events_completed_result)

In [38]:
events_test_median = events_completed_result.loc[0, "test_median"]
events_control_median = events_completed_result.loc[0, "control_median"]
events_diff = events_completed_result.loc[0, "median_difference"]
events_p = events_completed_result.loc[0, "p_value"]
events_significant = events_completed_result.loc[0, "significant_0.05"]

if events_significant and events_diff < 0:
    print("The Test group required significantly fewer events to complete the process.")
    print(f"Test median: {events_test_median:.2f} events | Control median: {events_control_median:.2f} events | Difference: {events_diff:.2f} events")
    print(f"The difference is statistically significant at alpha = 0.05, with p-value = {events_p:.4f}.")

elif events_diff > 0:
    print("The Test group did not reduce the number of events needed to complete the process.")
    print(f"Test median: {events_test_median:.2f} events | Control median: {events_control_median:.2f} events | Difference: {events_diff:.2f} events")
    print(f"The result goes in the opposite direction of the hypothesis, with p-value = {events_p:.4f}.")
    print("This suggests that completed Test sessions required more interactions than completed Control sessions.")

elif events_diff == 0:
    print("The Test group did not reduce the number of events needed to complete the process.")
    print(f"Test median: {events_test_median:.2f} events | Control median: {events_control_median:.2f} events | Difference: {events_diff:.2f} events")
    print(f"Both groups had the same median number of events per completed session, with p-value = {events_p:.4f}.")

else:
    print("The Test group did not show a statistically significant reduction in events per completed session.")
    print(f"Test median: {events_test_median:.2f} events | Control median: {events_control_median:.2f} events | Difference: {events_diff:.2f} events")
    print(f"p-value = {events_p:.4f}.")

The Test group did not reduce the number of events needed to complete the process.
Test median: 5.00 events | Control median: 5.00 events | Difference: 0.00 events
Both groups had the same median number of events per completed session, with p-value = 1.0000.


### Interpretation

The Test group did not reduce the number of events needed to complete the process.

Both groups had the same median number of events per completed session: 5 events in Test and 5 events in Control.

The Mann-Whitney U test does not provide evidence of a statistically significant reduction in the Test group, with p-value = 1.0000.

This suggests that although the new interface helped more users complete the process and reduced completion time, it did not reduce the number of interactions required to reach the confirmation step.

In [39]:
# =========================
# Combine all hypothesis test results
# =========================

final_results = pd.concat(hypothesis_results, ignore_index=True, sort=False)

final_results

,kpi,test_rate,control_rate,difference,z_stat,p_value,significant_0.05,test_type,alternative,decision,threshold,test_median,control_median,median_difference,u_stat,observed_direction
0,Completion Rate,0.5407,0.4826,0.0581,14.7129,0.0000,True,Two-proportion z-test,Test > Control,Reject H0,NaN,NaN,NaN,NaN,NaN,NaN
1,Completion Rate with 5% Threshold,0.5407,0.4826,0.0581,2.0497,0.0202,True,Two-proportion z-test with 5% threshold,Test - Control > 5%,Reject H0,0.05,NaN,NaN,NaN,NaN,NaN
2,Friction Rate,0.6805,0.6938,-0.0132,-3.6140,0.0002,True,Two-proportion z-test,Test < Control,Reject H0,NaN,NaN,NaN,NaN,NaN,NaN
3,Backtracking Rate,0.2977,0.2057,0.0920,26.7852,1.0000,False,Two-proportion z-test,Test < Control,Fail to reject H0,NaN,NaN,NaN,NaN,NaN,NaN
4,Repeated Steps Rate,0.4735,0.3780,0.0955,24.4340,1.0000,False,Two-proportion z-test,Test < Control,Fail to reject H0,NaN,NaN,NaN,NaN,NaN,NaN
5,Median Completion Time,NaN,NaN,NaN,NaN,0.0000,True,Mann-Whitney U test,Test < Control,Reject H0,NaN,3.95,4.5167,-0.5667,121519681.0,Test < Control
6,Median Events per Completed Session,NaN,NaN,NaN,NaN,1.0000,False,Mann-Whitney U test,Test < Control,Fail to reject H0,NaN,5.00,5.0000,0.0000,140078644.5,Test = Control


In [40]:
# =========================
# Standardize final results table
# =========================

final_results["test_value"] = final_results["test_rate"].combine_first(final_results["test_median"])
final_results["control_value"] = final_results["control_rate"].combine_first(final_results["control_median"])
final_results["difference_value"] = final_results["difference"].combine_first(final_results["median_difference"])

final_results["observed_direction"] = np.where(
    final_results["difference_value"] > 0,
    "Test > Control",
    np.where(final_results["difference_value"] < 0, "Test < Control", "Test = Control")
)

final_results["p_value_display"] = final_results["p_value"].apply(
    lambda x: "<0.0001" if x < 0.0001 else f"{x:.4f}"
)

final_results_clean = final_results[
    [
        "kpi",
        "test_type",
        "alternative",
        "observed_direction",
        "test_value",
        "control_value",
        "difference_value",
        "p_value_display",
        "decision",
        "significant_0.05"
    ]
].copy()

final_results_clean[["test_value", "control_value", "difference_value"]] = (
    final_results_clean[["test_value", "control_value", "difference_value"]].round(4)
)

final_results_clean

,kpi,test_type,alternative,observed_direction,test_value,control_value,difference_value,p_value_display,decision,significant_0.05
0,Completion Rate,Two-proportion z-test,Test > Control,Test > Control,0.5407,0.4826,0.0581,<0.0001,Reject H0,True
1,Completion Rate with 5% Threshold,Two-proportion z-test with 5% threshold,Test - Control > 5%,Test > Control,0.5407,0.4826,0.0581,0.0202,Reject H0,True
2,Friction Rate,Two-proportion z-test,Test < Control,Test < Control,0.6805,0.6938,-0.0132,0.0002,Reject H0,True
3,Backtracking Rate,Two-proportion z-test,Test < Control,Test > Control,0.2977,0.2057,0.0920,1.0000,Fail to reject H0,False
4,Repeated Steps Rate,Two-proportion z-test,Test < Control,Test > Control,0.4735,0.3780,0.0955,1.0000,Fail to reject H0,False
5,Median Completion Time,Mann-Whitney U test,Test < Control,Test < Control,3.9500,4.5167,-0.5667,<0.0001,Reject H0,True
6,Median Events per Completed Session,Mann-Whitney U test,Test < Control,Test = Control,5.0000,5.0000,0.0000,1.0000,Fail to reject H0,False


In [73]:
business_interpretations = {
    "Completion Rate": "Test improved the main success metric.",
    "Completion Rate with 5% Threshold": "Test passed the minimum practical improvement threshold.",
    "Friction Rate": "Test slightly reduced the broad friction rate, mainly by reducing abandonment.",
    "Backtracking Rate": "Test increased backtracking, suggesting more users moved backwards in the process.",
    "Repeated Steps Rate": "Test increased repeated steps, suggesting more repeated interactions or uncertainty.",
    "Median Completion Time": "Test completed sessions were faster than Control.",
    "Median Events per Completed Session": "Test did not reduce the number of events needed to complete."
}

final_results_clean["business_interpretation"] = final_results_clean["kpi"].map(business_interpretations)

final_results_clean

,kpi,test_type,hypothesis_direction,observed_result,test_value,control_value,difference_value,p_value_display,decision,significant_0.05,business_interpretation
0,Completion Rate,Two-proportion z-test,Test > Control,Test > Control,0.5407,0.4826,0.0581,<0.0001,Reject H0,True,Test improved the main success metric.
1,Completion Rate with 5% Threshold,Two-proportion z-test with 5% threshold,Test - Control > 5%,Test > Control,0.5407,0.4826,0.0581,0.0202,Reject H0,True,Test passed the minimum practical improvement ...
2,Friction Rate,Two-proportion z-test,Test < Control,Test < Control,0.6805,0.6938,-0.0132,0.0002,Reject H0,True,"Test slightly reduced the broad friction rate,..."
3,Backtracking Rate,Two-proportion z-test,Test < Control,Test > Control,0.2977,0.2057,0.0920,1.0000,Fail to reject H0,False,"Test increased backtracking, suggesting more u..."
4,Repeated Steps Rate,Two-proportion z-test,Test < Control,Test > Control,0.4735,0.3780,0.0955,1.0000,Fail to reject H0,False,"Test increased repeated steps, suggesting more..."
5,Median Completion Time,Mann-Whitney U test,Test < Control,Test < Control,3.9500,4.5167,-0.5667,<0.0001,Reject H0,True,Test completed sessions were faster than Control.
6,Median Events per Completed Session,Mann-Whitney U test,Test < Control,Test = Control,5.0000,5.0000,0.0000,1.0000,Fail to reject H0,False,Test did not reduce the number of events neede...


## Final KPI Results

The final results table summarizes the seven KPI tests used to evaluate the experiment.

The Test group improved the main success metric, Completion Rate, and passed the 5% practical improvement threshold.

However, the usability metrics show a more mixed result. Friction Rate was slightly lower in the Test group, but Backtracking Rate and Repeated Steps Rate were higher.

This suggests that the new interface helped more users complete the process, but it did not clearly create a smoother journey.

In [74]:
# =========================
# Save final results
# =========================

final_results_clean.to_csv(output_path / "ab_testing_results_summary.csv", index=False)

print("Final A/B testing results saved successfully.")

Final A/B testing results saved successfully.


## Funnel and Drop-off Analysis

After evaluating the main KPIs, a short funnel analysis is used to understand where users dropped off during the digital process.

This analysis is descriptive and is not treated as an additional hypothesis test. Its purpose is to support the interpretation of the A/B test results by showing how users moved through each step of the process.

In [44]:
# =========================
# Funnel and drop-off analysis
# =========================

funnel_steps = ["start", "step_1", "step_2", "step_3", "confirm"]

funnel_df = started_sessions_df.copy()

for step in funnel_steps:
    funnel_df[f"reached_{step}"] = funnel_df["step_sequence"].str.contains(step, regex=False, na=False)

funnel_summary = (
    funnel_df
    .groupby("variation")
    .agg(
        reached_start=("reached_start", "sum"),
        reached_step_1=("reached_step_1", "sum"),
        reached_step_2=("reached_step_2", "sum"),
        reached_step_3=("reached_step_3", "sum"),
        reached_confirm=("reached_confirm", "sum")
    )
)

funnel_summary

,reached_start,reached_step_1,reached_step_2,reached_step_3,reached_confirm
variation,,,,,
Control,30962,23328,19833,17882,14942
Test,33219,28083,24196,21694,17961


In [45]:
# =========================
# Prepare funnel table for readability
# =========================

funnel_long = (
    funnel_summary
    .reset_index()
    .melt(
        id_vars="variation",
        value_vars=[
            "reached_start",
            "reached_step_1",
            "reached_step_2",
            "reached_step_3",
            "reached_confirm"
        ],
        var_name="step",
        value_name="sessions"
    )
)

funnel_long["step"] = funnel_long["step"].str.replace("reached_", "", regex=False)

step_order = {
    "start": 1,
    "step_1": 2,
    "step_2": 3,
    "step_3": 4,
    "confirm": 5
}

funnel_long["step_order"] = funnel_long["step"].map(step_order)
funnel_long = funnel_long.sort_values(["variation", "step_order"])

funnel_long

,variation,step,sessions,step_order
0,Control,start,30962,1
2,Control,step_1,23328,2
4,Control,step_2,19833,3
6,Control,step_3,17882,4
8,Control,confirm,14942,5
1,Test,start,33219,1
3,Test,step_1,28083,2
5,Test,step_2,24196,3
7,Test,step_3,21694,4
9,Test,confirm,17961,5


In [46]:
# =========================
# Add conversion and drop-off rates
# =========================

funnel_long["previous_sessions"] = funnel_long.groupby("variation")["sessions"].shift(1)

funnel_long["step_conversion_rate"] = funnel_long["sessions"] / funnel_long["previous_sessions"]
funnel_long["step_dropoff_rate"] = 1 - funnel_long["step_conversion_rate"]

funnel_long.loc[funnel_long["step"] == "start", "step_conversion_rate"] = 1
funnel_long.loc[funnel_long["step"] == "start", "step_dropoff_rate"] = 0

funnel_long["overall_conversion_from_start"] = (
    funnel_long["sessions"] / funnel_long.groupby("variation")["sessions"].transform("first")
)

funnel_long[
    [
        "variation",
        "step",
        "sessions",
        "step_conversion_rate",
        "step_dropoff_rate",
        "overall_conversion_from_start"
    ]
].round(4)

,variation,step,sessions,step_conversion_rate,step_dropoff_rate,overall_conversion_from_start
0,Control,start,30962,1.0000,0.0000,1.0000
2,Control,step_1,23328,0.7534,0.2466,0.7534
4,Control,step_2,19833,0.8502,0.1498,0.6406
6,Control,step_3,17882,0.9016,0.0984,0.5775
8,Control,confirm,14942,0.8356,0.1644,0.4826
1,Test,start,33219,1.0000,0.0000,1.0000
3,Test,step_1,28083,0.8454,0.1546,0.8454
5,Test,step_2,24196,0.8616,0.1384,0.7284
7,Test,step_3,21694,0.8966,0.1034,0.6531
9,Test,confirm,17961,0.8279,0.1721,0.5407


In [47]:
funnel_display = (
    funnel_long
    .pivot(
        index="step",
        columns="variation",
        values=["sessions", "step_conversion_rate", "step_dropoff_rate", "overall_conversion_from_start"]
    )
    .round(4)
)

funnel_display

sessions          step_conversion_rate         step_dropoff_rate  \
variation  Control     Test              Control    Test           Control   
step                                                                         
confirm    14942.0  17961.0               0.8356  0.8279            0.1644   
start      30962.0  33219.0               1.0000  1.0000            0.0000   
step_1     23328.0  28083.0               0.7534  0.8454            0.2466   
step_2     19833.0  24196.0               0.8502  0.8616            0.1498   
step_3     17882.0  21694.0               0.9016  0.8966            0.0984   

                  overall_conversion_from_start          
variation    Test                       Control    Test  
step                                                     
confirm    0.1721                        0.4826  0.5407  
start      0.0000                        1.0000  1.0000  
step_1     0.1546                        0.7534  0.8454  
step_2     0.1384                        0.6406  0.7284  
step_3     0.1034                        0.5775  0.6531

### Interpretation

The funnel analysis shows how users moved through each step of the digital process.

This helps identify whether the Test group only improved final completion, or whether it also performed better at specific stages of the journey.

The funnel results should be interpreted as supporting evidence for the KPI analysis, not as separate hypothesis tests.

## Final Business Conclusion

The A/B test shows that the new interface was successful in improving the main business objective: increasing process completion.

The Test group achieved a higher Completion Rate than the Control group and passed the 5% practical improvement threshold. Completed sessions in the Test group were also faster, suggesting that users who completed the process were able to do so more quickly.

However, the results are not fully positive across all usability metrics. The Test group showed higher Backtracking Rate and higher Repeated Steps Rate, which suggests that some users may have experienced more uncertainty or repeated interactions during the process.

Overall, the redesigned interface appears to be effective at increasing completion, but it does not clearly deliver a smoother user journey. The recommendation would be to keep the new interface as the better-performing version, while investigating the causes of increased backtracking and repeated steps before a full rollout.

In [50]:
pd.read_csv("../1.3_KPI_AB_Testing_Outputs/ab_testing_results_summary.csv")

,kpi,test_type,alternative,observed_direction,test_value,control_value,difference_value,p_value_display,decision,significant_0.05,business_interpretation
0,Completion Rate,Two-proportion z-test,Test > Control,Test > Control,0.5407,0.4826,0.0581,<0.0001,Reject H0,True,Test improved the main success metric.
1,Completion Rate with 5% Threshold,Two-proportion z-test with 5% threshold,Test - Control > 5%,Test > Control,0.5407,0.4826,0.0581,0.0202,Reject H0,True,Test passed the minimum practical improvement ...
2,Friction Rate,Two-proportion z-test,Test < Control,Test < Control,0.6805,0.6938,-0.0132,0.0002,Reject H0,True,"Test slightly reduced friction, although frict..."
3,Backtracking Rate,Two-proportion z-test,Test < Control,Test > Control,0.2977,0.2057,0.0920,1.0000,Fail to reject H0,False,"Test increased backtracking, suggesting more u..."
4,Repeated Steps Rate,Two-proportion z-test,Test < Control,Test > Control,0.4735,0.3780,0.0955,1.0000,Fail to reject H0,False,"Test increased repeated steps, suggesting more..."
5,Median Completion Time,Mann-Whitney U test,Test < Control,Test < Control,3.9500,4.5167,-0.5667,<0.0001,Reject H0,True,Test completed sessions were faster than Control.
6,Median Events per Completed Session,Mann-Whitney U test,Test < Control,Test = Control,5.0000,5.0000,0.0000,1.0000,Fail to reject H0,False,Test did not reduce the number of events neede...


# Additional Analysis: Interpreting Ambiguous Results

The KPI results show a mixed but interesting outcome.

The Test version improved Completion Rate, passed the 5% threshold, and completed sessions were faster. However, the Test group also showed higher Backtracking Rate and higher Repeated Steps Rate.

This additional analysis explores these ambiguous results in more detail.

The goal is not to run more main hypothesis tests, but to better understand how the Test group achieved higher completion and whether those completions were smooth or involved more friction.

## 1. Clean vs Friction Completions

This analysis checks whether the higher Completion Rate in the Test group came from clean completions or from completions that still involved friction.

A clean completion is a completed session that followed the ideal journey:

`start → step_1 → step_2 → step_3 → confirm`

A friction completion is a completed session that reached `confirm`, but did not follow the ideal journey. This may include repeated steps, backtracking, skipped steps, or other non-linear behavior.

In [58]:
# =========================
# Clean vs friction completions
# =========================

completion_quality_df = started_sessions_df.copy()

completion_quality_df["clean_completion"] = (
    completion_quality_df["completed"] & completion_quality_df["straight_through"]
)

completion_quality_df["friction_completion"] = (
    completion_quality_df["completed"] & ~completion_quality_df["straight_through"]
)

completion_quality_summary = (
    completion_quality_df
    .groupby("variation")
    .agg(
        started_sessions=("visit_id", "count"),
        completed_sessions=("completed", "sum"),
        clean_completions=("clean_completion", "sum"),
        friction_completions=("friction_completion", "sum")
    )
)

completion_quality_summary["completion_rate"] = (
    completion_quality_summary["completed_sessions"] / completion_quality_summary["started_sessions"]
)

completion_quality_summary["clean_completion_rate"] = (
    completion_quality_summary["clean_completions"] / completion_quality_summary["started_sessions"]
)

completion_quality_summary["friction_completion_rate"] = (
    completion_quality_summary["friction_completions"] / completion_quality_summary["started_sessions"]
)

completion_quality_summary["clean_share_of_completions"] = (
    completion_quality_summary["clean_completions"] / completion_quality_summary["completed_sessions"]
)

completion_quality_summary["friction_share_of_completions"] = (
    completion_quality_summary["friction_completions"] / completion_quality_summary["completed_sessions"]
)

completion_quality_summary = completion_quality_summary.round(4)

completion_quality_summary

,started_sessions,completed_sessions,clean_completions,friction_completions,completion_rate,clean_completion_rate,friction_completion_rate,clean_share_of_completions,friction_share_of_completions
variation,,,,,,,,,
Control,30962,14942,9482,5460,0.4826,0.3062,0.1763,0.6346,0.3654
Test,33219,17961,10613,7348,0.5407,0.3195,0.2212,0.5909,0.4091


In [59]:
# =========================
# Test vs Control differences
# =========================

completion_quality_diff = (
    completion_quality_summary.loc["Test"] - completion_quality_summary.loc["Control"]
).to_frame(name="test_minus_control")

completion_quality_diff = completion_quality_diff.loc[
    [
        "completion_rate",
        "clean_completion_rate",
        "friction_completion_rate",
        "clean_share_of_completions",
        "friction_share_of_completions"
    ]
].round(4)

completion_quality_diff

,test_minus_control
completion_rate,0.0581
clean_completion_rate,0.0133
friction_completion_rate,0.0449
clean_share_of_completions,-0.0437
friction_share_of_completions,0.0437


In [60]:
test_completion_rate = completion_quality_summary.loc["Test", "completion_rate"]
control_completion_rate = completion_quality_summary.loc["Control", "completion_rate"]

test_clean_rate = completion_quality_summary.loc["Test", "clean_completion_rate"]
control_clean_rate = completion_quality_summary.loc["Control", "clean_completion_rate"]

test_friction_rate = completion_quality_summary.loc["Test", "friction_completion_rate"]
control_friction_rate = completion_quality_summary.loc["Control", "friction_completion_rate"]

print("Clean vs Friction Completion Summary")
print("-" * 40)
print(f"Completion Rate - Test: {test_completion_rate:.2%} | Control: {control_completion_rate:.2%}")
print(f"Clean Completion Rate - Test: {test_clean_rate:.2%} | Control: {control_clean_rate:.2%}")
print(f"Friction Completion Rate - Test: {test_friction_rate:.2%} | Control: {control_friction_rate:.2%}")

if test_clean_rate > control_clean_rate and test_friction_rate > control_friction_rate:
    print("\nThe Test group increased both clean completions and friction completions.")
elif test_clean_rate > control_clean_rate and test_friction_rate <= control_friction_rate:
    print("\nThe Test group mainly improved completion through cleaner journeys.")
elif test_clean_rate <= control_clean_rate and test_friction_rate > control_friction_rate:
    print("\nThe Test group mainly improved completion through completions that still involved friction.")
else:
    print("\nThe Test group did not clearly improve either clean or friction completions.")

Clean vs Friction Completion Summary
----------------------------------------
Completion Rate - Test: 54.07% | Control: 48.26%
Clean Completion Rate - Test: 31.95% | Control: 30.62%
Friction Completion Rate - Test: 22.12% | Control: 17.63%

The Test group increased both clean completions and friction completions.


### Interpretation

The Test group increased overall Completion Rate from 48.26% to 54.07%, an improvement of 5.81 percentage points.

However, this improvement did not come mainly from clean journeys.

Clean Completion Rate increased only slightly, from 30.62% in Control to 31.95% in Test, a difference of 1.33 percentage points.

By contrast, Friction Completion Rate increased from 17.63% in Control to 22.12% in Test, a difference of 4.49 percentage points.

This suggests that the Test interface helped more users reach the confirmation step, but many of the additional completions still involved repeated steps, backtracking, skipped steps, or other non-linear behavior.

Among completed sessions, the share of clean completions was lower in Test than in Control: 59.09% versus 63.46%. This reinforces the idea that the redesign improved completion, but not necessarily the smoothness of the user journey.

## 2. Backtracking and Repeated Steps Among Completed Sessions

The previous analysis showed that the Test group increased both clean completions and friction completions.

This section focuses only on completed sessions to understand whether successful Test journeys were smoother or more complex than successful Control journeys.

The goal is to check whether completed Test sessions still involved more backtracking and repeated steps.

In [61]:
# =========================
# Backtracking and repeated steps among completed sessions
# =========================

completed_quality_summary = (
    completed_sessions_df
    .groupby("variation")
    .agg(
        completed_sessions=("visit_id", "count"),
        backtracking_completed_sessions=("has_backtracking", "sum"),
        repeated_step_completed_sessions=("has_repeated_steps", "sum"),
        step_jump_completed_sessions=("has_step_jump", "sum"),
        clean_completed_sessions=("straight_through", "sum"),
        median_completion_time_min=("completion_time_min", "median"),
        median_events=("n_events", "median")
    )
)

completed_quality_summary["backtracking_rate_completed"] = (
    completed_quality_summary["backtracking_completed_sessions"] / completed_quality_summary["completed_sessions"]
)

completed_quality_summary["repeated_steps_rate_completed"] = (
    completed_quality_summary["repeated_step_completed_sessions"] / completed_quality_summary["completed_sessions"]
)

completed_quality_summary["step_jump_rate_completed"] = (
    completed_quality_summary["step_jump_completed_sessions"] / completed_quality_summary["completed_sessions"]
)

completed_quality_summary["clean_completion_share"] = (
    completed_quality_summary["clean_completed_sessions"] / completed_quality_summary["completed_sessions"]
)

completed_quality_summary = completed_quality_summary.round(4)

completed_quality_summary

,completed_sessions,backtracking_completed_sessions,repeated_step_completed_sessions,step_jump_completed_sessions,clean_completed_sessions,median_completion_time_min,median_events,backtracking_rate_completed,repeated_steps_rate_completed,step_jump_rate_completed,clean_completion_share
variation,,,,,,,,,,,
Control,14942,3329,5403,134,9482,4.5167,5.0,0.2228,0.3616,0.0090,0.6346
Test,17961,4164,7311,175,10613,3.9500,5.0,0.2318,0.4070,0.0097,0.5909


In [62]:
# =========================
# Test vs Control differences among completed sessions
# =========================

completed_quality_diff = (
    completed_quality_summary.loc["Test"] - completed_quality_summary.loc["Control"]
).to_frame(name="test_minus_control")

completed_quality_diff = completed_quality_diff.loc[
    [
        "backtracking_rate_completed",
        "repeated_steps_rate_completed",
        "step_jump_rate_completed",
        "clean_completion_share",
        "median_completion_time_min",
        "median_events"
    ]
].round(4)

completed_quality_diff

,test_minus_control
backtracking_rate_completed,0.0090
repeated_steps_rate_completed,0.0454
step_jump_rate_completed,0.0007
clean_completion_share,-0.0437
median_completion_time_min,-0.5667
median_events,0.0000


In [63]:
test_completed_backtracking = completed_quality_summary.loc["Test", "backtracking_rate_completed"]
control_completed_backtracking = completed_quality_summary.loc["Control", "backtracking_rate_completed"]

test_completed_repeated = completed_quality_summary.loc["Test", "repeated_steps_rate_completed"]
control_completed_repeated = completed_quality_summary.loc["Control", "repeated_steps_rate_completed"]

test_clean_share = completed_quality_summary.loc["Test", "clean_completion_share"]
control_clean_share = completed_quality_summary.loc["Control", "clean_completion_share"]

print("Completed Session Quality Summary")
print("-" * 40)
print(f"Backtracking among completed sessions - Test: {test_completed_backtracking:.2%} | Control: {control_completed_backtracking:.2%}")
print(f"Repeated steps among completed sessions - Test: {test_completed_repeated:.2%} | Control: {control_completed_repeated:.2%}")
print(f"Clean share among completed sessions - Test: {test_clean_share:.2%} | Control: {control_clean_share:.2%}")

if test_completed_backtracking > control_completed_backtracking and test_completed_repeated > control_completed_repeated:
    print("\nCompleted Test sessions involved more backtracking and repeated steps than completed Control sessions.")
elif test_completed_backtracking <= control_completed_backtracking and test_completed_repeated <= control_completed_repeated:
    print("\nCompleted Test sessions were at least as smooth as completed Control sessions on these two metrics.")
else:
    print("\nThe completed session quality results are mixed across backtracking and repeated steps.")

Completed Session Quality Summary
----------------------------------------
Backtracking among completed sessions - Test: 23.18% | Control: 22.28%
Repeated steps among completed sessions - Test: 40.70% | Control: 36.16%
Clean share among completed sessions - Test: 59.09% | Control: 63.46%

Completed Test sessions involved more backtracking and repeated steps than completed Control sessions.


### Interpretation

Among completed sessions, the Test group was faster but not cleaner.

The median Completion Time was lower in Test than in Control: 3.95 minutes compared with 4.52 minutes. This means completed Test sessions were around 34 seconds faster.

However, completed Test sessions showed slightly more Backtracking and clearly more Repeated Steps. Backtracking among completed sessions increased from 22.28% in Control to 23.18% in Test, while Repeated Steps increased from 36.16% to 40.70%.

The share of clean completions was also lower in Test: 59.09% compared with 63.46% in Control.

This suggests that the redesigned interface helped users complete the process faster and more often, but many successful Test journeys still involved repeated or non-linear behavior.

## 3. Funnel and Drop-off Analysis

This funnel analysis shows how users moved through each step of the digital process.

The goal is to understand where users dropped off and whether the Test group performed better across the journey or mainly at the final confirmation stage.

This analysis is descriptive and is not treated as an additional hypothesis test.

In [64]:
# =========================
# Build funnel step flags
# =========================

funnel_steps = ["start", "step_1", "step_2", "step_3", "confirm"]

funnel_df = started_sessions_df.copy()

for step in funnel_steps:
    funnel_df[f"reached_{step}"] = funnel_df["step_sequence"].str.contains(step, regex=False, na=False)

funnel_df[[f"reached_{step}" for step in funnel_steps]].head()

,reached_start,reached_step_1,reached_step_2,reached_step_3,reached_confirm
0,True,True,True,True,True
1,True,True,True,True,True
2,True,False,False,False,False
3,True,True,True,True,False
4,True,False,False,False,False


In [65]:
# =========================
# Create funnel summary
# =========================

funnel_summary = (
    funnel_df
    .groupby("variation")
    .agg(
        reached_start=("reached_start", "sum"),
        reached_step_1=("reached_step_1", "sum"),
        reached_step_2=("reached_step_2", "sum"),
        reached_step_3=("reached_step_3", "sum"),
        reached_confirm=("reached_confirm", "sum")
    )
)

funnel_summary

,reached_start,reached_step_1,reached_step_2,reached_step_3,reached_confirm
variation,,,,,
Control,30962,23328,19833,17882,14942
Test,33219,28083,24196,21694,17961


In [66]:
# =========================
# Convert funnel table to long format
# =========================

funnel_long = (
    funnel_summary
    .reset_index()
    .melt(
        id_vars="variation",
        value_vars=[
            "reached_start",
            "reached_step_1",
            "reached_step_2",
            "reached_step_3",
            "reached_confirm"
        ],
        var_name="step",
        value_name="sessions"
    )
)

funnel_long["step"] = funnel_long["step"].str.replace("reached_", "", regex=False)

step_order = {"start": 1, "step_1": 2, "step_2": 3, "step_3": 4, "confirm": 5}

funnel_long["step_order"] = funnel_long["step"].map(step_order)
funnel_long = funnel_long.sort_values(["variation", "step_order"]).reset_index(drop=True)

funnel_long

,variation,step,sessions,step_order
0,Control,start,30962,1
1,Control,step_1,23328,2
2,Control,step_2,19833,3
3,Control,step_3,17882,4
4,Control,confirm,14942,5
5,Test,start,33219,1
6,Test,step_1,28083,2
7,Test,step_2,24196,3
8,Test,step_3,21694,4
9,Test,confirm,17961,5


In [67]:
# =========================
# Add conversion and drop-off rates
# =========================

funnel_long["previous_sessions"] = funnel_long.groupby("variation")["sessions"].shift(1)

funnel_long["step_conversion_rate"] = funnel_long["sessions"] / funnel_long["previous_sessions"]
funnel_long["step_dropoff_rate"] = 1 - funnel_long["step_conversion_rate"]

funnel_long.loc[funnel_long["step"] == "start", "step_conversion_rate"] = 1
funnel_long.loc[funnel_long["step"] == "start", "step_dropoff_rate"] = 0

funnel_long["overall_conversion_from_start"] = (
    funnel_long["sessions"] / funnel_long.groupby("variation")["sessions"].transform("first")
)

funnel_long_display = funnel_long[
    [
        "variation",
        "step",
        "sessions",
        "step_conversion_rate",
        "step_dropoff_rate",
        "overall_conversion_from_start"
    ]
].copy()

funnel_long_display[
    [
        "step_conversion_rate",
        "step_dropoff_rate",
        "overall_conversion_from_start"
    ]
] = funnel_long_display[
    [
        "step_conversion_rate",
        "step_dropoff_rate",
        "overall_conversion_from_start"
    ]
].round(4)

funnel_long_display

,variation,step,sessions,step_conversion_rate,step_dropoff_rate,overall_conversion_from_start
0,Control,start,30962,1.0000,0.0000,1.0000
1,Control,step_1,23328,0.7534,0.2466,0.7534
2,Control,step_2,19833,0.8502,0.1498,0.6406
3,Control,step_3,17882,0.9016,0.0984,0.5775
4,Control,confirm,14942,0.8356,0.1644,0.4826
5,Test,start,33219,1.0000,0.0000,1.0000
6,Test,step_1,28083,0.8454,0.1546,0.8454
7,Test,step_2,24196,0.8616,0.1384,0.7284
8,Test,step_3,21694,0.8966,0.1034,0.6531
9,Test,confirm,17961,0.8279,0.1721,0.5407


In [68]:
# =========================
# Compare Test vs Control by funnel step
# =========================

funnel_comparison = (
    funnel_long_display
    .pivot(
        index="step",
        columns="variation",
        values="overall_conversion_from_start"
    )
)

funnel_comparison["test_minus_control"] = funnel_comparison["Test"] - funnel_comparison["Control"]

funnel_comparison = funnel_comparison.loc[["start", "step_1", "step_2", "step_3", "confirm"]].round(4)

funnel_comparison

variation,Control,Test,test_minus_control
step,,,
start,1.0000,1.0000,0.0000
step_1,0.7534,0.8454,0.0920
step_2,0.6406,0.7284,0.0878
step_3,0.5775,0.6531,0.0756
confirm,0.4826,0.5407,0.0581


### Interpretation

The funnel analysis shows that the Test group performed better mainly in the earlier stages of the journey.

From `start` to `step_1`, Test retained 84.54% of started sessions, compared with 75.34% in Control. This is the largest difference in the funnel, with Test outperforming Control by 9.20 percentage points.

Test also maintained a higher overall conversion from start through the intermediate steps. By `step_3`, 65.31% of Test sessions had reached that stage, compared with 57.75% of Control sessions.

However, the final step from `step_3` to `confirm` was slightly weaker in Test. Test converted 82.79% of sessions from `step_3` to `confirm`, compared with 83.56% in Control.

This means that the higher overall Completion Rate in Test was mainly driven by stronger retention earlier in the funnel, not by better conversion at the final confirmation step.

This supports the broader conclusion: the redesigned interface helped more users progress through the process, especially at the beginning, but it did not necessarily make every part of the journey smoother.

## 4. Friction Rate Decomposition

## 4. Why Did Friction Rate Improve While Backtracking and Repeated Steps Worsened?

The KPI results show an apparent contradiction.

The Test group had a slightly lower Friction Rate, but higher Backtracking Rate and higher Repeated Steps Rate.

This happens because Friction Rate is a broad metric. It captures any session that did not follow the ideal straight-through journey, including both incomplete sessions and completed sessions with non-linear behavior.

Therefore, Friction Rate can improve if fewer users abandon the process, even if some specific friction behaviors, such as backtracking or repeated steps, increase.

In [69]:
# =========================
# Friction Rate decomposition
# =========================

friction_decomposition_df = started_sessions_df.copy()

friction_decomposition_df["incomplete_session"] = ~friction_decomposition_df["completed"]

friction_decomposition_df["clean_completion"] = (
    friction_decomposition_df["completed"] & friction_decomposition_df["straight_through"]
)

friction_decomposition_df["friction_completion"] = (
    friction_decomposition_df["completed"] & ~friction_decomposition_df["straight_through"]
)

friction_decomposition = (
    friction_decomposition_df
    .groupby("variation")
    .agg(
        started_sessions=("visit_id", "count"),
        completed_sessions=("completed", "sum"),
        incomplete_sessions=("incomplete_session", "sum"),
        clean_completions=("clean_completion", "sum"),
        friction_completions=("friction_completion", "sum"),
        friction_sessions=("friction_flag", "sum"),
        backtracking_sessions=("has_backtracking", "sum"),
        repeated_step_sessions=("has_repeated_steps", "sum")
    )
)

friction_decomposition["completion_rate"] = (
    friction_decomposition["completed_sessions"] / friction_decomposition["started_sessions"]
)

friction_decomposition["incomplete_rate"] = (
    friction_decomposition["incomplete_sessions"] / friction_decomposition["started_sessions"]
)

friction_decomposition["clean_completion_rate"] = (
    friction_decomposition["clean_completions"] / friction_decomposition["started_sessions"]
)

friction_decomposition["friction_completion_rate"] = (
    friction_decomposition["friction_completions"] / friction_decomposition["started_sessions"]
)

friction_decomposition["friction_rate"] = (
    friction_decomposition["friction_sessions"] / friction_decomposition["started_sessions"]
)

friction_decomposition["backtracking_rate"] = (
    friction_decomposition["backtracking_sessions"] / friction_decomposition["started_sessions"]
)

friction_decomposition["repeated_steps_rate"] = (
    friction_decomposition["repeated_step_sessions"] / friction_decomposition["started_sessions"]
)

friction_decomposition.round(4)

,started_sessions,completed_sessions,incomplete_sessions,clean_completions,friction_completions,friction_sessions,backtracking_sessions,repeated_step_sessions,completion_rate,incomplete_rate,clean_completion_rate,friction_completion_rate,friction_rate,backtracking_rate,repeated_steps_rate
variation,,,,,,,,,,,,,,,
Control,30962,14942,16020,9482,5460,21480,6368,11703,0.4826,0.5174,0.3062,0.1763,0.6938,0.2057,0.3780
Test,33219,17961,15258,10613,7348,22606,9889,15728,0.5407,0.4593,0.3195,0.2212,0.6805,0.2977,0.4735


In [70]:
# =========================
# Test minus Control differences
# =========================

friction_decomposition_diff = (
    friction_decomposition.loc["Test"] - friction_decomposition.loc["Control"]
).to_frame(name="test_minus_control")

friction_decomposition_diff = friction_decomposition_diff.loc[
    [
        "completion_rate",
        "incomplete_rate",
        "clean_completion_rate",
        "friction_completion_rate",
        "friction_rate",
        "backtracking_rate",
        "repeated_steps_rate"
    ]
].round(4)

friction_decomposition_diff

,test_minus_control
completion_rate,0.0581
incomplete_rate,-0.0581
clean_completion_rate,0.0132
friction_completion_rate,0.0449
friction_rate,-0.0132
backtracking_rate,0.0920
repeated_steps_rate,0.0955


In [71]:
# =========================
# Composition of broad friction
# =========================

friction_composition = friction_decomposition.copy()

friction_composition["incomplete_share_of_friction"] = (
    friction_composition["incomplete_sessions"] / friction_composition["friction_sessions"]
)

friction_composition["friction_completion_share_of_friction"] = (
    friction_composition["friction_completions"] / friction_composition["friction_sessions"]
)

friction_composition["backtracking_share_of_friction"] = (
    friction_composition["backtracking_sessions"] / friction_composition["friction_sessions"]
)

friction_composition["repeated_steps_share_of_friction"] = (
    friction_composition["repeated_step_sessions"] / friction_composition["friction_sessions"]
)

friction_composition_display = friction_composition[
    [
        "incomplete_share_of_friction",
        "friction_completion_share_of_friction",
        "backtracking_share_of_friction",
        "repeated_steps_share_of_friction"
    ]
].round(4)

friction_composition_display

,incomplete_share_of_friction,friction_completion_share_of_friction,backtracking_share_of_friction,repeated_steps_share_of_friction
variation,,,,
Control,0.7458,0.2542,0.2965,0.5448
Test,0.6750,0.3250,0.4375,0.6957


Backtracking and repeated steps can overlap with other friction categories, so these shares should not be expected to sum to 100%.

### Interpretation

This decomposition explains the apparent contradiction between Friction Rate, Backtracking Rate, and Repeated Steps Rate.

At first glance, the Test group appears slightly better on Friction Rate: 68.05% compared with 69.38% in Control, a reduction of 1.32 percentage points.

However, Friction Rate is a broad metric. It captures every session that did not follow a perfect straight-through journey, including both incomplete sessions and completed sessions with non-linear behavior.

The Test group reduced the Incomplete Rate from 51.74% to 45.93%, a decrease of 5.81 percentage points. This means fewer users abandoned the process.

However, the Test group also increased Friction Completion Rate from 17.63% to 22.12%, an increase of 4.49 percentage points. This means more users reached the confirmation step, but many of them did so through non-clean journeys.

The navigation-specific friction metrics also worsened. Backtracking Rate increased from 20.57% to 29.77%, and Repeated Steps Rate increased from 37.80% to 47.35%.

This suggests that the new interface did not simply reduce friction. A more accurate interpretation is that the Test version reduced abandonment, but increased navigation friction.

In other words, the redesign helped more users reach the end of the process, but many of those additional completions involved repeated steps, backtracking, or other non-linear behavior.

## Final Business Conclusion

The A/B test shows that the new interface was successful in improving the main business objective: increasing process completion.

The Test group achieved a higher Completion Rate than the Control group and passed the 5% practical improvement threshold. Completed Test sessions were also faster, suggesting that users who reached the confirmation step were able to complete the process more quickly.

However, the improvement was not a clean usability win. The additional analysis showed that the Test version reduced abandonment, but increased navigation friction. Backtracking Rate and Repeated Steps Rate were both higher in the Test group, and a larger share of Test completions involved non-clean journeys.

Overall, the redesigned interface appears to be more effective at getting users to the end of the process, but not necessarily smoother. The recommendation would be to keep the Test version as the stronger-performing base, while investigating why users repeated steps and moved backwards more often before a full rollout.